[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/grego921/studio-video-ai/blob/main/Viggle_Studio_Colab.ipynb)

# ⚡ STUDIO VIDEO AI — MOTOR VIGGLE-ANIMATE (COMFYUI)
**Ponte de Alta Performance com GPU NVIDIA A100 / L4 (Google Colab)**

Este notebook inicializa o motor neural **Viggle-Animate (MiniMax-H3)** via ComfyUI integrado ao painel web **Studio Video AI**.
Ele executa:
1. Conexão com Google Drive para salvar os modelos (~25 GB) de forma permanente;
2. Diagnóstico de GPU e trava de segurança para VRAM >= 22 GB;
3. Instalação do ComfyUI e dos nós customizados (`ComfyUI-Viggle-Animate-H3`, `ComfyUI-VideoHelperSuite`, `ComfyUI-KJNodes`, etc.);
4. Download dos modelos para o Google Drive e links simbólicos com o ComfyUI;
5. Inicialização do ComfyUI local (`127.0.0.1:8188`) sem `--lowvram`;
6. Inicialização da API REST `colab_viggle_api.py` na porta `7860`;
7. Túnel público Cloudflare seguro (`trycloudflare.com`) para conectar ao site na Vercel.


## 0. Montar Google Drive (Armazenamento dos Modelos)
Monta o Google Drive para armazenar permanentemente os modelos do Viggle-Animate (~25 GB).
Nas próximas vezes que você abrir este notebook, os modelos já estarão no seu Drive e não precisarão ser baixados novamente!


In [ ]:
# ==============================================================================
# 0. CONECTAR GOOGLE DRIVE (PERSISTÊNCIA DOS MODELOS)
# ==============================================================================
import os

print("📁 Conectando ao Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive montado com sucesso em /content/drive!")
except ImportError:
    print("ℹ️ Executando fora do Google Colab (ambiente local ou de testes).")

DRIVE_MODELS_DIR = "/content/drive/MyDrive/StudioVideoAI/models"
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
print(f"✅ Pasta de modelos no Google Drive pronta:\n   {DRIVE_MODELS_DIR}")


## 1. Diagnóstico de Hardware (GPU e VRAM)
Verifica a presença da GPU e confirma se há pelo menos 22 GB de VRAM (ex.: A100 ou L4).


In [ ]:
# ==============================================================================
# 1. DIAGNÓSTICO DE HARDWARE: GPU E VRAM
# ==============================================================================
import sys
import torch

print("🔍 Verificando GPU disponível no ambiente...")

if not torch.cuda.is_available():
    print("\n" + "="*70)
    print("❌ ERRO: Nenhuma GPU detectada!")
    print("No menu do Colab, vá em: Ambiente de execução -> Alterar tipo de ambiente de execução")
    print("e selecione uma GPU (L4 ou A100 no Colab Pro).")
    print("="*70 + "\n")
    raise RuntimeError("Nenhuma GPU ativa no Google Colab.")

gpu_name = torch.cuda.get_device_name(0)
total_vram_bytes = torch.cuda.get_device_properties(0).total_memory
total_vram_gb = total_vram_bytes / (1024 ** 3)

print("\n" + "="*70)
print(f"🎮 GPU Detectada : {gpu_name}")
print(f"💾 VRAM Total    : {total_vram_gb:.2f} GB")
print("="*70 + "\n")

# Exibe tabela do nvidia-smi
!nvidia-smi

# Verificação de VRAM mínima de segurança (22 GB)
if total_vram_gb < 21.5:
    print("\n" + "!"*70)
    print("🛑 BLOQUEIO DE CAPACIDADE DE MEMÓRIA VRAM:")
    print("O menor modelo tem 21 GB; use L4 ou A100 (Colab Pro)")
    print("!"*70 + "\n")
    raise RuntimeError("O menor modelo tem 21 GB; use L4 ou A100 (Colab Pro)")

print("\n✅ VRAM suficiente detectada! Você pode prosseguir para a próxima célula.")


## 2. Instalação do ComfyUI, Custom Nodes e Dependências
Instala o ComfyUI oficial, os nós do Viggle-Animate, VideoHelperSuite, KJNodes, ferramentas de mídia (ffmpeg, edge-tts, opencv) e dependências.


In [ ]:
# ==============================================================================
# 2. INSTALAÇÃO DO COMFYUI E CUSTOM NODES
# ==============================================================================
import os
import subprocess

print("📦 1/3 Instalando ComfyUI oficial...")
%cd /content
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub flask flask-cors edge-tts opencv-python pillow

print("\n🧩 2/3 Clonando e configurando Custom Nodes...")
custom_nodes_dir = "/content/ComfyUI/custom_nodes"
os.makedirs(custom_nodes_dir, exist_ok=True)
%cd {custom_nodes_dir}

# 1. ComfyUI-Viggle-Animate-H3 (Saganaki22)
if not os.path.exists("ComfyUI-Viggle-Animate-H3"):
    !git clone https://github.com/Saganaki22/ComfyUI-Viggle-Animate-H3.git
if os.path.exists("ComfyUI-Viggle-Animate-H3/requirements.txt"):
    !pip install -q -r ComfyUI-Viggle-Animate-H3/requirements.txt

# 2. ComfyUI-VideoHelperSuite (VHS - Kosinkadink)
if not os.path.exists("ComfyUI-VideoHelperSuite"):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
if os.path.exists("ComfyUI-VideoHelperSuite/requirements.txt"):
    !pip install -q -r ComfyUI-VideoHelperSuite/requirements.txt

# 3. ComfyUI-KJNodes (necessário para os nós do workflow: CustomSigmas, ModelPreviewOverrideKJ)
if not os.path.exists("ComfyUI-KJNodes"):
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git
if os.path.exists("ComfyUI-KJNodes/requirements.txt"):
    !pip install -q -r ComfyUI-KJNodes/requirements.txt

# 4. comfyui-various (para o nó JWImageResizeToMegapixels presente no workflow)
if not os.path.exists("comfyui-various"):
    !git clone https://github.com/jamesWalker55/comfyui-various.git

%cd /content
print("\n✅ ComfyUI e nós customizados instalados com sucesso!")


## 3. Download e Persistência dos Modelos no Google Drive
Verifica o espaço livre no Google Drive, baixa os modelos oficiais apenas se estiverem ausentes ou incompletos, e cria links simbólicos para a pasta do ComfyUI.


In [ ]:
# ==============================================================================
# 3. DOWNLOAD E PERSISTÊNCIA DOS MODELOS NO GOOGLE DRIVE
# ==============================================================================
import os
import shutil
import urllib.request
import time
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()

DRIVE_MODELS_DIR = "/content/drive/MyDrive/StudioVideoAI/models"
COMFYUI_MODELS_DIR = "/content/ComfyUI/models"

os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
os.makedirs(COMFYUI_MODELS_DIR, exist_ok=True)

DOWNLOAD_LIST = [
    {
        "name": "Diffusion Model (int8 pruned, ~21 GB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "diffusion_models/minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors",
        "subfolder": "diffusion_models",
        "target_file": "minimax_h3_ref2va_viggle_pruned_int8_convrot.safetensors",
        "expected_size": 21033720080  # bytes (~21.03 GB)
    },
    {
        "name": "LoRA DMD 4-Step (r64, ~0.94 GB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "loras/viggle_animate_dmd_lora_r64.safetensors",
        "subfolder": "loras",
        "target_file": "viggle_animate_dmd_lora_r64.safetensors",
        "expected_size": 941893032  # bytes (~0.94 GB)
    },
    {
        "name": "Text Conditioning Embeddings (~3.7 MB)",
        "repo_id": "drbaph/Viggle-Animate-ComfyUI",
        "repo_file": "text_cond/fixed_embed_fwd_anyframe.safetensors",
        "subfolder": "text_cond",
        "target_file": "fixed_embed_fwd_anyframe.safetensors",
        "expected_size": 3713456  # bytes (~3.71 MB)
    },
    {
        "name": "Video VAE int8 convrot (~3.17 GB)",
        "repo_id": "Kijai/MiniMax-H3-experimental",
        "repo_file": "minimax_h3_video_vae_int8_convrot.safetensors",
        "subfolder": "vae",
        "target_file": "minimax_h3_video_vae_int8_convrot.safetensors",
        "expected_size": 3171670912  # bytes (~3.17 GB)
    }
]

print("🔍 1/3 Verificando status dos modelos no Google Drive...\n")

bytes_needed = 0
for item in DOWNLOAD_LIST:
    drive_sub_dir = os.path.join(DRIVE_MODELS_DIR, item["subfolder"])
    os.makedirs(drive_sub_dir, exist_ok=True)
    drive_file_path = os.path.join(drive_sub_dir, item["target_file"])

    if os.path.exists(drive_file_path):
        actual_size = os.path.getsize(drive_file_path)
        if actual_size == item["expected_size"]:
            item["status"] = "present"
            continue
        else:
            print(f"⚠️ {item['target_file']} no Drive com tamanho incorreto ({actual_size} bytes vs {item['expected_size']} esperados).")
            item["status"] = "invalid_size"
    else:
        item["status"] = "missing"

    bytes_needed += item["expected_size"]

# 2. Verificar espaço livre no Drive antes de baixar
if bytes_needed > 0:
    print(f"📊 Espaço necessário para download: {bytes_needed / (1024**3):.2f} GB")
    try:
        drive_stat = shutil.disk_usage(DRIVE_MODELS_DIR)
        free_bytes = drive_stat.free
    except Exception:
        free_bytes = shutil.disk_usage("/content/drive").free

    free_gb = free_bytes / (1024**3)
    needed_gb = bytes_needed / (1024**3)

    if free_bytes < bytes_needed:
        missing_bytes = bytes_needed - free_bytes
        missing_gb = missing_bytes / (1024**3)
        print("\n" + "=" * 75)
        print("❌ ESPAÇO INSUFICIENTE NO GOOGLE DRIVE!")
        print("=" * 75)
        print(f"Espaço livre no Google Drive : {free_gb:.2f} GB")
        print(f"Espaço total necessário      : {needed_gb:.2f} GB")
        print(f"FALTAM NO SEU GOOGLE DRIVE    : {missing_gb:.2f} GB")
        print("=" * 75)
        print(f"Por favor, libere pelo menos {missing_gb:.2f} GB no seu Google Drive e execute novamente.")
        print("=" * 75 + "\n")
        raise RuntimeError(
            f"Espaço insuficiente no Google Drive. Disponível: {free_gb:.2f} GB, "
            f"Necessário: {needed_gb:.2f} GB. Faltam {missing_gb:.2f} GB no seu Google Drive."
        )
    else:
        print(f"✅ Espaço livre verificado no Drive: {free_gb:.2f} GB disponíveis (suficiente).\n")

# 3. Baixar modelos para o Drive (apenas os ausentes ou com tamanho incorreto)
print("⬇️ 2/3 Processando downloads para o Google Drive...\n")

for item in DOWNLOAD_LIST:
    drive_sub_dir = os.path.join(DRIVE_MODELS_DIR, item["subfolder"])
    drive_dest_path = os.path.join(drive_sub_dir, item["target_file"])

    if item.get("status") == "present":
        print(f"⏩ [PULADO] {item['target_file']} já está no Drive, pulando.")
    else:
        print(f"📡 Baixando {item['name']}...")
        print(f"   Origem: {item['repo_id']} / {item['repo_file']}")
        
        temp_cache = "/content/hf_cache"
        os.makedirs(temp_cache, exist_ok=True)
        downloaded = hf_hub_download(
            repo_id=item["repo_id"],
            filename=item["repo_file"],
            local_dir=temp_cache,
            local_dir_use_symlinks=False
        )
        
        print(f"🚚 Salvando no Google Drive: {drive_dest_path}...")
        if os.path.exists(drive_dest_path):
            os.remove(drive_dest_path)
        shutil.move(downloaded, drive_dest_path)
        print(f"✅ Concluído: {item['target_file']} no Drive ({os.path.getsize(drive_dest_path) / (1024**3):.2f} GB).\n")

# 4. Conectar ComfyUI ao Drive com links simbólicos
print("\n🔗 3/3 Conectando pastas de modelos do ComfyUI ao Google Drive com links simbólicos...")

for item in DOWNLOAD_LIST:
    drive_dest_path = os.path.join(DRIVE_MODELS_DIR, item["subfolder"], item["target_file"])
    comfy_sub_dir = os.path.join(COMFYUI_MODELS_DIR, item["subfolder"])
    os.makedirs(comfy_sub_dir, exist_ok=True)
    comfy_dest_path = os.path.join(comfy_sub_dir, item["target_file"])

    if os.path.islink(comfy_dest_path) or os.path.exists(comfy_dest_path):
        try:
            if os.path.islink(comfy_dest_path) and os.path.realpath(comfy_dest_path) == os.path.realpath(drive_dest_path):
                print(f"🔗 Link já configurado: ComfyUI/models/{item['subfolder']}/{item['target_file']}")
                continue
            os.remove(comfy_dest_path)
        except Exception:
            pass

    os.symlink(drive_dest_path, comfy_dest_path)
    print(f"✅ Link criado: ComfyUI/models/{item['subfolder']}/{item['target_file']} -> Drive")

# Baixar os workflows API base do repositório GitHub
os.makedirs("/content/workflows", exist_ok=True)
wf_dest = "/content/workflows/viggle_api.json"
if not os.path.exists(wf_dest) or os.path.getsize(wf_dest) < 100:
    wf_url = f"https://raw.githubusercontent.com/grego921/studio-video-ai/main/workflows/viggle_api.json?t={int(time.time())}"
    print("\n📥 Baixando workflow base viggle_api.json do GitHub...")
    urllib.request.urlretrieve(wf_url, wf_dest)

wf_longo_dest = "/content/workflows/viggle_longo_api.json"
if not os.path.exists(wf_longo_dest) or os.path.getsize(wf_longo_dest) < 100:
    try:
        wf_longo_url = f"https://raw.githubusercontent.com/grego921/studio-video-ai/main/workflows/viggle_longo_api.json?t={int(time.time())}"
        print("📥 Baixando workflow base viggle_longo_api.json do GitHub...")
        urllib.request.urlretrieve(wf_longo_url, wf_longo_dest)
    except Exception as e_wf_longo:
        print(f"⚠️ Fallback de viggle_longo_api.json não disponível no GitHub ({e_wf_longo}); será gerado na célula 5.")

print("\n🎉 Todos os modelos no Drive e links simbólicos do ComfyUI estão prontos!")


## 4. Iniciar o ComfyUI (Modo Alta Performance)
Inicia o ComfyUI em segundo plano sem `--lowvram` (aproveitando os 40 GB/80 GB da A100 para máxima velocidade).


In [ ]:
# ==============================================================================
# 4. INICIALIZAÇÃO DO COMFYUI (SEM --lowvram)
# ==============================================================================
import os
import time
import subprocess
import urllib.request

def print_comfyui_log_tail(log_path="/content/comfyui.log", num_lines=40):
    print("\n" + "=" * 75)
    print(f"📋 ÚLTIMAS {num_lines} LINHAS DE {log_path}:")
    print("=" * 75)
    if os.path.exists(log_path):
        try:
            with open(log_path, "r", encoding="utf-8", errors="replace") as f:
                lines = f.readlines()
                tail = lines[-num_lines:] if len(lines) > num_lines else lines
                print("".join(tail).strip())
        except Exception as e:
            print(f"Não foi possível ler o arquivo de log ({e})")
    else:
        print("Arquivo de log não encontrado.")
    print("=" * 75 + "\n")

!pkill -f "python main.py" || true
time.sleep(1)

print("🚀 Iniciando ComfyUI oficial na porta 8188 (sem --lowvram)...")
comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=comfy_log,
    stderr=subprocess.STDOUT
)

print("⏳ Aguardando ComfyUI inicializar e responder na porta 8188 (tempo limite: 300 s)...")
comfy_ready = False
max_wait_seconds = 300
start_time = time.time()

while time.time() - start_time < max_wait_seconds:
    # 1. Verifica se o processo do ComfyUI encerrou prematuramente
    if comfy_proc.poll() is not None:
        print(f"\n❌ O processo do ComfyUI encerrou prematuramente (código de saída: {comfy_proc.returncode}).")
        print_comfyui_log_tail()
        raise RuntimeError(
            f"O processo do ComfyUI encerrou prematuramente com código {comfy_proc.returncode}. "
            "Consulte as últimas linhas do log acima para diagnóstico."
        )

    # 2. Testa requisição HTTP no endpoint /system_stats
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1.5) as resp:
            if resp.status == 200:
                comfy_ready = True
                elapsed_total = int(time.time() - start_time)
                print(f"✅ ComfyUI ativo e pronto para receber tarefas! ({elapsed_total} s)")
                break
    except Exception:
        pass

    elapsed = int(time.time() - start_time)
    if elapsed > 0 and elapsed % 15 == 0:
        print(f"⏳ Aguardando inicialização do ComfyUI ({elapsed}/{max_wait_seconds} s)...")

    time.sleep(1)

if not comfy_ready:
    print("\n❌ Tempo limite de 300 s atingido sem resposta do ComfyUI.")
    print_comfyui_log_tail()
    raise TimeoutError(
        "Tempo limite de 300 s excedido aguardando o ComfyUI responder na porta 8188. "
        "Consulte as últimas linhas do log acima para diagnóstico."
    )


## 5. Gerar Workflow API Oficial, Iniciar API REST e Túnel Cloudflared
Baixa a versão mais recente dos scripts do GitHub, gera o workflow API oficial em `/content/workflows/viggle_api.json` a partir do ComfyUI em execução (porta 8188) com `build_viggle_workflow.py`, inicia o servidor Flask na porta `7860` e expõe via túnel público seguro do Cloudflare.


In [ ]:
# ==============================================================================
# 5. GERAR WORKFLOW API OFICIAL, INICIAR SERVIDOR API E TÚNEL CLOUDFLARE
# ==============================================================================
import os
import re
import sys
import json
import time
import subprocess
import urllib.request
import urllib.error

# 0. Verificação prévia de execução da Célula 4
if not os.path.exists("/content/comfyui.log"):
    print("\n" + "=" * 75)
    print("❌ A célula 4 não foi executada nesta sessão. Rode a célula 4 primeiro.")
    print("=" * 75 + "\n")
    raise RuntimeError("A célula 4 não foi executada nesta sessão. Rode a célula 4 primeiro.")

def print_failure_and_exit(title: str, details: str = "", log_path: str = None, num_lines: int = 40):
    if log_path and os.path.exists(log_path):
        print("\n" + "=" * 75)
        print(f"📋 ÚLTIMAS {num_lines} LINHAS DE {log_path}:")
        print("=" * 75)
        try:
            with open(log_path, "r", encoding="utf-8", errors="replace") as f:
                lines = f.readlines()
                print("".join(lines[-num_lines:]).strip())
        except Exception as e:
            print(f"Não foi possível ler o arquivo de log ({e})")
        print("=" * 75)

    print("\n" + "!" * 75)
    print(f"❌ DIAGNÓSTICO DE FALHA: {title}")
    print("!" * 75)
    if details:
        print(f"\n{details.strip()}\n")
    print("=" * 75)
    print("🚨 COPIE TUDO DAQUI PARA CIMA E ENVIE AO SUPORTE")
    print("=" * 75 + "\n")
    raise RuntimeError(f"FALHA: {title}")

# 1. Instalar Cloudflared se necessário
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 2. Atualizar scripts do GitHub raw (branch main) com cache-buster
print("🔄 Baixando scripts atualizados do GitHub (branch main)...")
t_now = int(time.time())
api_url = f"https://raw.githubusercontent.com/grego921/studio-video-ai/main/colab_viggle_api.py?t={t_now}"
urllib.request.urlretrieve(api_url, "/content/colab_viggle_api.py")

build_url = f"https://raw.githubusercontent.com/grego921/studio-video-ai/main/build_viggle_workflow.py?t={t_now}"
urllib.request.urlretrieve(build_url, "/content/build_viggle_workflow.py")

# 3. Aguardar ComfyUI responder em /system_stats por até 300 s antes de gerar o workflow
print("\n⏳ Verificando disponibilidade do ComfyUI em http://127.0.0.1:8188/system_stats (até 300 s)...")
comfy_ready_5 = False
max_wait_c5 = 300
start_c5 = time.time()

while time.time() - start_c5 < max_wait_c5:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2.0) as resp:
            if resp.status == 200:
                comfy_ready_5 = True
                elapsed_c5 = int(time.time() - start_c5)
                print(f"✅ ComfyUI respondendo normalmente na porta 8188! ({elapsed_c5} s)")
                break
    except Exception:
        pass

    elapsed = int(time.time() - start_c5)
    if elapsed > 0 and elapsed % 15 == 0:
        print(f"⏳ Aguardando ComfyUI responder em /system_stats ({elapsed}/{max_wait_c5} s)...")

    time.sleep(1)

if not comfy_ready_5:
    print_failure_and_exit(
        title="ComfyUI não respondeu na porta 8188 após 300 s",
        details="O ComfyUI não respondeu em http://127.0.0.1:8188/system_stats dentro do prazo de 300 segundos.",
        log_path="/content/comfyui.log",
        num_lines=40
    )

# 4. Gerar workflows API a partir do ComfyUI em execução (sobrescrevendo os esboços)
print("\n⚙️ Gerando workflow API oficial (curto) a partir do ComfyUI em execução (porta 8188)...")
cmd_build = [
    "python", "/content/build_viggle_workflow.py",
    "--workflow-in", "/content/ComfyUI/custom_nodes/ComfyUI-Viggle-Animate-H3/example_workflows/viggle-animate-h3_workflow-v1.2.0.json",
    "--workflow-out", "/content/workflows/viggle_api.json",
    "--comfy-url", "http://127.0.0.1:8188"
]
res_build = subprocess.run(cmd_build, capture_output=True, text=True)
print(res_build.stdout)
if res_build.returncode != 0:
    print_failure_and_exit(
        title="Erro ao gerar o workflow API do Viggle-Animate",
        details=res_build.stderr or res_build.stdout,
        log_path="/content/comfyui.log",
        num_lines=40
    )

longo_example = "/content/ComfyUI/custom_nodes/ComfyUI-Viggle-Animate-H3/example_workflows/viggle-animate-h3_workflow-chunked-sampler.json"
if not os.path.exists(longo_example):
    print("📥 Exemplo chunked-sampler não encontrado localmente; baixando do GitHub...")
    longo_example = "/content/workflows/viggle-animate-h3_workflow-chunked-sampler.json"
    os.makedirs("/content/workflows", exist_ok=True)
    try:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/grego921/studio-video-ai/main/workflows/viggle-animate-h3_workflow-chunked-sampler.json?t={int(time.time())}",
            longo_example
        )
    except Exception as e_dl_longo:
        print_failure_and_exit(
            title="Falha ao baixar viggle-animate-h3_workflow-chunked-sampler.json do GitHub",
            details=str(e_dl_longo),
            log_path="/content/comfyui.log",
            num_lines=40
        )

print("\n⚙️ Gerando workflow API oficial (longo / chunked-sampler) a partir do ComfyUI em execução (porta 8188)...")
cmd_build_longo = [
    "python", "/content/build_viggle_workflow.py",
    "--workflow-in", longo_example,
    "--workflow-out", "/content/workflows/viggle_longo_api.json",
    "--comfy-url", "http://127.0.0.1:8188"
]
res_build_longo = subprocess.run(cmd_build_longo, capture_output=True, text=True)
print(res_build_longo.stdout)
if res_build_longo.returncode != 0:
    print_failure_and_exit(
        title="Erro ao gerar o workflow API longo do Viggle-Animate",
        details=res_build_longo.stderr or res_build_longo.stdout,
        log_path="/content/comfyui.log",
        num_lines=40
    )

# 5. Encerrar processos anteriores
!pkill -f cloudflared || true
!pkill -f colab_viggle_api.py || true
time.sleep(1)

# 6. Iniciar colab_viggle_api.py na porta 7860 e aguardar até 60 s pelo health check
print("🚀 Iniciando colab_viggle_api.py na porta 7860...")
api_log = open('/content/viggle_api.log', 'w')
env = os.environ.copy()
env["PORT"] = "7860"
env["COMFYUI_DIR"] = "/content/ComfyUI"
env["WORKFLOW_API_PATH"] = "/content/workflows/viggle_api.json"
env["WORKFLOW_LONGO_API_PATH"] = "/content/workflows/viggle_longo_api.json"

api_proc = subprocess.Popen(
    ['python', '-u', '/content/colab_viggle_api.py'],
    stdout=api_log,
    stderr=subprocess.STDOUT,
    env=env
)

print("⏳ Aguardando servidor colab_viggle_api.py responder em http://127.0.0.1:7860/api/health (até 60 s)...")
api_ready = False
api_timeout_s = 60
start_api = time.time()

while time.time() - start_api < api_timeout_s:
    # Verifica se o processo da API encerrou inesperadamente
    if api_proc.poll() is not None:
        print_failure_and_exit(
            title=f"O servidor colab_viggle_api.py encerrou inesperadamente com código {api_proc.returncode}",
            details="O processo da API Flask na porta 7860 foi finalizado antes de aceitar conexões.",
            log_path="/content/viggle_api.log",
            num_lines=40
        )

    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/api/health", timeout=2.0) as resp:
            if resp.status == 200:
                api_ready = True
                elapsed_api = int(time.time() - start_api)
                print(f"✅ Servidor colab_viggle_api.py ativo na porta 7860! ({elapsed_api} s)")
                break
    except Exception:
        pass

    elapsed = int(time.time() - start_api)
    if elapsed > 0 and elapsed % 15 == 0:
        print(f"⏳ Aguardando API responder ({elapsed}/{api_timeout_s} s)...")

    time.sleep(1)

if not api_ready:
    print_failure_and_exit(
        title="Tempo limite de 60 s excedido aguardando o servidor colab_viggle_api.py responder",
        details="A rota http://127.0.0.1:7860/api/health não respondeu dentro de 60 segundos.",
        log_path="/content/viggle_api.log",
        num_lines=40
    )

# 7. Iniciar Túnel Cloudflared (espera até 90 s, até 3 tentativas)
print("\n🌐 Abrindo túnel público seguro do Cloudflare...")
tunnel_url = None
max_tunnel_attempts = 3
tunnel_wait_s = 90

for attempt in range(1, max_tunnel_attempts + 1):
    print(f"\n🚀 [Tentativa {attempt}/{max_tunnel_attempts}] Iniciando cloudflared tunnel (tempo limite: {tunnel_wait_s} s)...")
    !pkill -f cloudflared || true
    time.sleep(1)

    tunnel_log = open('/content/tunnel.log', 'w')
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
        stdout=tunnel_log,
        stderr=subprocess.STDOUT
    )

    start_tun = time.time()
    while time.time() - start_tun < tunnel_wait_s:
        if os.path.exists('/content/tunnel.log'):
            try:
                with open('/content/tunnel.log', 'r', encoding='utf-8', errors='replace') as f:
                    content = f.read()
                    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
                    if match:
                        tunnel_url = match.group(0)
                        break
            except Exception:
                pass

        if tunnel_proc.poll() is not None and not tunnel_url:
            break

        time.sleep(1)

    if tunnel_url:
        print(f"✅ URL do túnel obtida com sucesso: {tunnel_url}")
        break
    else:
        print(f"\n⚠️ [Tentativa {attempt}/{max_tunnel_attempts}] Falha ao obter URL do túnel em {tunnel_wait_s} s.")
        if os.path.exists('/content/tunnel.log'):
            print("\n" + "=" * 75)
            print("📋 ÚLTIMAS 40 LINHAS DE /content/tunnel.log:")
            print("=" * 75)
            try:
                with open('/content/tunnel.log', 'r', encoding='utf-8', errors='replace') as f:
                    lines = f.readlines()
                    print("".join(lines[-40:]).strip())
            except Exception:
                pass
            print("=" * 75 + "\n")
        !pkill -f cloudflared || true
        try:
            tunnel_proc.terminate()
            tunnel_proc.wait(timeout=2)
        except Exception:
            pass
        time.sleep(2)

if not tunnel_url:
    print_failure_and_exit(
        title=f"Falha ao abrir o túnel Cloudflare após {max_tunnel_attempts} tentativas de {tunnel_wait_s} s",
        details="O cloudflared não conseguiu estabelecer uma URL pública *.trycloudflare.com para a porta 7860.",
        log_path="/content/tunnel.log",
        num_lines=40
    )

# 8. Teste de ponta a ponta: GET {tunnel_url}/api/health
print(f"\n🔍 Verificando integridade da conexão pública em {tunnel_url}/api/health (até 120 s)...")
public_ready = False
health_data = {}
max_pub_wait_s = 120
start_pub = time.time()

while time.time() - start_pub < max_pub_wait_s:
    elapsed = int(time.time() - start_pub)
    print(f"⏳ Aguardando o Cloudflare ativar a rota ({elapsed}/{max_pub_wait_s} s)...")
    try:
        req = urllib.request.Request(f"{tunnel_url}/api/health", headers={"User-Agent": "ColabHealthCheck/1.0"})
        with urllib.request.urlopen(req, timeout=5.0) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode('utf-8'))
                if data.get("ready") is True:
                    health_data = data
                    public_ready = True
                    break
    except Exception:
        pass

    time.sleep(3)

# Checagem da API local para diagnóstico
local_health_data = {}
local_ready = False
try:
    with urllib.request.urlopen("http://127.0.0.1:7860/api/health", timeout=3.0) as resp:
        if resp.status == 200:
            local_health_data = json.loads(resp.read().decode('utf-8'))
            if local_health_data.get("ready") is True:
                local_ready = True
except Exception:
    local_ready = False

# Se a API local NÃO estiver com ready=true, levanta erro com o JSON do health
if not local_ready:
    problems = local_health_data.get("workflow_problems", [])
    missing_ckpts = [
        k for k in ["checkpoint_diffusion", "checkpoint_lora", "checkpoint_text_cond", "checkpoint_vae"]
        if local_health_data.get(k) is not True
    ]
    missing_other = [
        k for k in ["cuda_available", "vram_ok", "workflow_exists", "workflow_valid", "comfy_online"]
        if local_health_data.get(k) is not True
    ]

    details_parts = [
        f"Retorno do /api/health local (ready={local_health_data.get('ready', False)}):",
        json.dumps(local_health_data, indent=2, ensure_ascii=False) if local_health_data else "A API local (http://127.0.0.1:7860/api/health) não respondeu."
    ]
    if missing_ckpts:
        details_parts.append("\n🚨 Checkpoints ausentes:\n  - " + "\n  - ".join(missing_ckpts))
    if missing_other:
        details_parts.append("\n🚨 Requisitos do sistema não atendidos:\n  - " + "\n  - ".join(missing_other))
    if problems:
        details_parts.append("\n🚨 Problemas detectados no workflow (workflow_problems):\n  - " + "\n  - ".join(problems))

    print_failure_and_exit(
        title="A API local não respondeu com ready=true",
        details="\n".join(details_parts),
        log_path="/content/viggle_api.log",
        num_lines=40
    )

# Se a API local está ready=true mas a URL pública não respondeu em 120 s: NÃO levantar erro
if not public_ready:
    print("\n" + "=" * 75)
    print("⚠️ AVISO: TÚNEL CRIADO, AGUARDANDO ATIVAÇÃO DE BORDA")
    print(f"👉 URL DO MOTOR GPU : {tunnel_url}")
    print("=" * 75)
    print("O túnel foi criado mas ainda não respondeu. Aguarde 1 minuto e tente conectar no site. Se não funcionar, rode esta célula de novo.")
    print("=" * 75)
    print(f"🎮 GPU DETECTADA    : {local_health_data.get('gpu', 'NVIDIA')}")
    print(f"⚡ MOTOR NEURAL     : {local_health_data.get('engine', 'Viggle-Animate H3')}")
    print("=" * 75 + "\n")
    print("1. Copie a URL acima;")
    print("2. No site (painel web), clique no botão 'Conectar Motor GPU';")
    print("3. Cole a URL e clique em 'Salvar e Conectar';")
    print("4. Mantenha esta célula executando durante o uso da ferramenta.")
else:
    # SUCESSO COMPLETO CONFIRMADO
    display_data = health_data if health_data else local_health_data
    print("\n" + "=" * 75)
    print("🎉 MOTOR GPU VIGGLE-ANIMATE CONECTADO COM SUCESSO!")
    print(f"👉 URL DO MOTOR GPU : {tunnel_url}")
    print(f"🎮 GPU DETECTADA    : {display_data.get('gpu', 'NVIDIA')}")
    print(f"⚡ MOTOR NEURAL     : {display_data.get('engine', 'Viggle-Animate H3')}")
    print("=" * 75 + "\n")
    print("1. Copie a URL acima;")
    print("2. No site (painel web), clique no botão 'Conectar Motor GPU';")
    print("3. Cole a URL e clique em 'Salvar e Conectar';")
    print("4. Mantenha esta célula executando durante o uso da ferramenta.")
